# NRP-Exp: Verification of Manuscript Tables

This notebook independently reconstructs and verifies the numerical content reported in Tables 3, 4, 6, 7, and 8 of the NRP-Exp manuscript.

The notebook distinguishes between:

- Source-instance information obtained directly from the retained JSON files.
- Experimental configurations declared for the reported executions.
- Structural properties observed after preprocessing.
- Plans returned by the configured solvers.
- Distinct objective vectors represented by those plans.
- Solver-portability results for the controlled linear formulation.

Table 4 is treated as a retained experimental configuration rather than as a result derived from the source instances. The remaining tables are reconstructed from the retained artefacts and executable framework components.

Software version: v1.2.0

This notebook is intended to verify the retained manuscript tables against the archived implementation and retained datasets.

The checks are bounded and focus on:

- source-instance reconstruction;
- preprocessing effects;
- execution of retained configurations;
- controlled solver-portability checks.

The notebook does not constitute a formal verification of the framework.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    """Locate the project root by searching for the src directory."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing the 'src' directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\imagu\OneDrive\Escritorio\NRPExperimentFramework


## Import

In [2]:
import json
import platform
import time
from collections import Counter

import numpy as np
import pandas as pd
import scipy

from src.domain.core.loader import GenericJSONLoader
from src.domain.core.pipeline import PreprocessingEngine
from src.domain.plugins.nrp import NRPPlugin

from src.solving.core.problem import (
    ConstraintSpec,
    OptimizationProblem,
    ThresholdOperator,
)

from src.solving.algorithms.brute_force import BruteForceSolver
from src.solving.algorithms.greedy import GreedySolver
from src.solving.algorithms.milp import MILPSolver
from src.solving.algorithms.nsga2 import NSGA2Solver
from src.solving.algorithms.random import RandomSearchSolver
from src.solving.algorithms.simulated_annealing import (
    SimulatedAnnealingSolver,
)


environment = {
    "Python": platform.python_version(),
    "Platform": platform.platform(),
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "SciPy": scipy.__version__,
}

pd.Series(
    environment,
    name="Version",
).to_frame()

,Version
Python,3.13.14
Platform,Windows-11-10.0.26300-SP0
NumPy,2.4.6
Pandas,3.0.3
SciPy,1.18.0


## Additional functions

In [9]:
def locate_instance(*candidate_names: str) -> Path:
    """
    Locate an instance under examples/instances using one of the
    supplied file names.
    """
    instance_root = PROJECT_ROOT / "examples" / "instances"

    for candidate_name in candidate_names:
        matches = list(
            instance_root.rglob(candidate_name)
        )

        if matches:
            return matches[0]

    raise FileNotFoundError(
        f"Could not locate any of: {candidate_names}"
    )


def load_case(path: Path):
    """Load both the raw JSON data and the framework model."""
    raw_text = path.read_text(encoding="utf-8")
    raw_data = json.loads(raw_text)

    model = GenericJSONLoader.load(
        raw_text,
        NRPPlugin,
    )

    return raw_data, model


def relationship_count(raw_data, name):
    """Count a relationship type in either supported location."""
    relationships = raw_data.get("relationships", {}) or {}

    return len(
        relationships.get(name, [])
        or raw_data.get(name, [])
        or []
    )


def get_value_dependencies(raw_data):
    relationships = raw_data.get("relationships", {}) or {}

    return (
        relationships.get("value_dependencies", [])
        or raw_data.get("value_dependencies", [])
        or []
    )


def unique_objective_vectors(front, objective_names, digits=6):
    """
    Return the distinct objective vectors represented by a Pareto front.
    """
    return {
        tuple(
            round(
                float(solution.objectives[name]),
                digits,
            )
            for name in objective_names
        )
        for solution in front.solutions
    }


def unique_release_compositions(front):
    """Return the distinct selected-item compositions."""
    return {
        tuple(
            sorted(
                str(item_id)
                for item_id in solution.selected_ids
            )
        )
        for solution in front.solutions
    }


def count_dynamic_value_dependencies(model):
    return len(
        model.relationships.get(
            "value_dependencies",
            [],
        )
    )


def branch_label(model):
    return getattr(
        model,
        "branch_name",
        "Base",
    )

def select_variant(processed_models, requested_name):
    """
    Select a processed model by branch name.

    'Base' selects the sole/base model.
    """
    if requested_name == "Base":
        if len(processed_models) != 1:
            raise ValueError(
                "Base was requested, but preprocessing generated "
                f"{len(processed_models)} variants."
            )

        return processed_models[0]

    matching = [
        model
        for model in processed_models
        if branch_label(model) == requested_name
    ]

    if len(matching) != 1:
        available = [
            branch_label(model)
            for model in processed_models
        ]

        raise ValueError(
            f"Expected one variant named {requested_name!r}; "
            f"found {len(matching)}. Available: {available}"
        )

    return matching[0]

def build_reported_solver(config):
    """
    Build the solver declared in Table 4.
    """

    solver_name = config["solver"]
    parameters = config["parameters"]

    if solver_name == "Brute Force":
        return BruteForceSolver(**parameters)

    if solver_name == "NSGA-II":
        return NSGA2Solver(**parameters)

    raise ValueError(
        f"Unsupported solver: {solver_name}"
    )

## Validation case inventory

The following mapping connects the case labels used in the manuscript with the corresponding retained JSON files.

If a local file uses a different name, only the candidate file names in the mapping below must be adjusted

In [4]:
CASE_FILES = {
    "Motorola": (
        "Motorola_35.json",
        "motorola_35.json",
    ),
    "Classic": (
        "Bagnall_20.json",
        "Classic_20.json",
    ),
    "Classic-100": (
        "Sagrado_100_corrected.json",
        "Sagrado_100.json",
        "Classic_100.json",
    ),
    "WordProc-50": (
        "WordProc_50.json",
        "wordproc_50.json",
    ),
    "Concise_Case_11": (
        "Concise_Case_11.json",
    ),
    "Long_Case_30": (
        "Long_Case_30.json",
    ),
    "Controlled_Linear_6": (
        "Controlled_Linear_6.json",
        "controlled_linear_6.json",
    ),
}


CASE_PATHS = {
    case_name: locate_instance(*candidate_names)
    for case_name, candidate_names
    in CASE_FILES.items()
}


pd.Series(
    {
        case_name: str(path.relative_to(PROJECT_ROOT))
        for case_name, path in CASE_PATHS.items()
    },
    name="JSON path",
).to_frame()

,JSON path
Motorola,examples\instances\paper\Motorola_35.json
Classic,examples\instances\paper\Bagnall_20.json
Classic-100,examples\instances\paper\Sagrado_100.json
WordProc-50,examples\instances\paper\WordProc_50.json
Concise_Case_11,examples\instances\paper\Concise_Case_11.json
Long_Case_30,examples\instances\paper\Long_Case_30.json
Controlled_Linear_6,examples\instances\paper\Controlled_Linear_6.json


## Table 3

In [5]:
TABLE3_OBJECTIVE_COUNTS = {
    "Motorola": 2,
    "Classic": 2,
    "Classic-100": 2,
    "WordProc-50": 3,
    "Concise_Case_11": 2,
    "Long_Case_30": 2,
    "Controlled_Linear_6": 2,
}


table3_rows = []

for case_name, path in CASE_PATHS.items():
    raw_data, _ = load_case(path)

    requirements = (
        raw_data.get("requirements")
        or raw_data.get("items")
        or []
    )

    evaluator_count = (
        len(raw_data.get("stakeholders", []))
        + len(raw_data.get("developers", []))
    )

    # Table 3 uses the principal evaluator/stakeholder count retained
    # for the source profile. For Concise, use stakeholders only.
    if raw_data.get("stakeholders") is not None:
        stakeholder_count = len(
            raw_data.get("stakeholders", [])
        )
    else:
        stakeholder_count = evaluator_count

    table3_rows.append(
        {
            "Case": case_name,
            "Stakeholders": stakeholder_count,
            "Requirements": len(requirements),
            "Objectives": TABLE3_OBJECTIVE_COUNTS[case_name],
            "Bounds": 1,
            "Implications": relationship_count(
                raw_data,
                "precedences",
            ),
            "Couplings": relationship_count(
                raw_data,
                "couplings",
            ),
            "Exclusions": relationship_count(
                raw_data,
                "exclusions",
            ),
            "Value-based interactions": len(
                get_value_dependencies(raw_data)
            ),
        }
    )


table3_observed = pd.DataFrame(table3_rows)

table3_observed

,Case,Stakeholders,Requirements,Objectives,Bounds,Implications,Couplings,Exclusions,Value-based interactions
0,Motorola,4,35,2,1,0,0,0,0
1,Classic,5,20,2,1,8,2,0,0
2,Classic-100,5,100,2,1,38,4,0,0
3,WordProc-50,4,50,3,1,66,8,0,7
4,Concise_Case_11,2,12,2,1,11,1,1,3
5,Long_Case_30,4,30,2,1,26,5,3,6
6,Controlled_Linear_6,0,6,2,1,0,0,0,0


## Table 4: Retained experimental configurations
Table 4 is a configuration table rather than a derived result.

The cell below records the exact objectives, resource bound, selected solver, selected model variant, and principal solver parameters used for each reported execution.

The table can be used both as:

an executable configuration registry; and
a direct check against the manuscript.

In [6]:
TABLE4_CONFIG = {
    "Motorola": {
        "objectives": {
            "satisfaction": "max",
            "effort": "min",
        },
        "constraint_attribute": "effort",
        "constraint_value": 1415,
        "solver": "NSGA-II",
        "parameters": {
            "pop_size": 80,
            "generations": 150,
        },
        "variant": "Base",
    },
    "Classic": {
        "objectives": {
            "satisfaction": "max",
            "effort": "min",
        },
        "constraint_attribute": "effort",
        "constraint_value": 43,
        "solver": "Brute Force",
        "parameters": {
            "max_evaluations": 1_000_000,
        },
        "variant": "Base",
    },
    "Classic-100": {
        "objectives": {
            "satisfaction": "max",
            "effort": "min",
        },
        "constraint_attribute": "effort",
        "constraint_value": 519,
        "solver": "NSGA-II",
        "parameters": {
            "pop_size": 100,
            "generations": 150,
        },
        "variant": "Base",
    },
    "WordProc-50": {
        "objectives": {
            "value": "max",
            "urgency": "max",
            "effort": "min",
        },
        "constraint_attribute": "effort",
        "constraint_value": 725,
        "solver": "NSGA-II",
        "parameters": {
            "pop_size": 80,
            "generations": 150,
        },
        "variant": "Base",
    },
    "Concise_Case_11": {
        "objectives": {
            "satisfaction": "max",
            "effort": "min",
        },
        "constraint_attribute": "effort",
        "constraint_value": 28,
        "solver": "Brute Force",
        "parameters": {
            "max_evaluations": 30_000,
        },
        "variant": "Excl(r7)",
    },
    "Long_Case_30": {
        "objectives": {
            "satisfaction": "max",
            "effort": "min",
        },
        "constraint_attribute": "effort",
        "constraint_value": 85,
        "solver": "NSGA-II",
        "parameters": {
            "pop_size": 80,
            "generations": 150,
        },
        "variant": "Excl(r9) ➔ Excl(r17)",
    },
}


table4_display = pd.DataFrame(
    [
        {
            "Case": case_name,
            "Objectives": ", ".join(
                f"{name} ({direction})"
                for name, direction
                in config["objectives"].items()
            ),
            "Bound": (
                f"{config['constraint_attribute']} "
                f"<= {config['constraint_value']}"
            ),
            "Solver": config["solver"],
            "Parameters": config["parameters"],
            "Variant": config["variant"],
        }
        for case_name, config in TABLE4_CONFIG.items()
    ]
)

table4_display

,Case,Objectives,Bound,Solver,Parameters,Variant
0,Motorola,"satisfaction (max), effort (min)",effort <= 1415,NSGA-II,"{'pop_size': 80, 'generations': 150}",Base
1,Classic,"satisfaction (max), effort (min)",effort <= 43,Brute Force,{'max_evaluations': 1000000},Base
2,Classic-100,"satisfaction (max), effort (min)",effort <= 519,NSGA-II,"{'pop_size': 100, 'generations': 150}",Base
3,WordProc-50,"value (max), urgency (max), effort (min)",effort <= 725,NSGA-II,"{'pop_size': 80, 'generations': 150}",Base
4,Concise_Case_11,"satisfaction (max), effort (min)",effort <= 28,Brute Force,{'max_evaluations': 30000},Excl(r7)
5,Long_Case_30,"satisfaction (max), effort (min)",effort <= 85,NSGA-II,"{'pop_size': 80, 'generations': 150}",Excl(r9) ➔ Excl(r17)


## Table 6: Structural effects of preprocessing
For each source instance, the complete preprocessing pipeline is executed. The resulting model variants are summarised using:

- source requirements;
- number of generated variants;
- active decision-item range;
- reduction range;
- remaining implications;
- unresolved couplings;
- unresolved exclusions;
- dynamically retained value-based interactions.

Ranges are reported when exclusion branching generates variants with different structural sizes.

In [7]:
def format_range(values):
    values = sorted(set(values))

    if len(values) == 1:
        return values[0]

    return f"{values[0]}-{values[-1]}"


def format_percentage_range(values):
    values = sorted(
        set(
            round(float(value), 2)
            for value in values
        )
    )
    if not values:
        return ""
        
    if len(values) == 1:
        return f"{values[0]:.2f}%"

    return f"{values[0]:.2f}-{values[-1]:.2f}%"

table6_rows = []
processed_case_models = {}

for case_name in TABLE4_CONFIG:
    raw_data, source_model = load_case(
        CASE_PATHS[case_name]
    )

    processed_models = PreprocessingEngine.process(
        initial_model=source_model,
        pipeline_options={},
    )

    processed_case_models[case_name] = processed_models

    source_items = len(source_model.items)
    active_counts = [
        len(model.items)
        for model in processed_models
    ]

    reductions = [
        100.0 * (source_items - active) / source_items
        for active in active_counts
    ]

    implication_counts = [
        len(
            model.relationships.get(
                "precedences",
                [],
            )
        )
        for model in processed_models
    ]

    coupling_counts = [
        len(
            model.relationships.get(
                "couplings",
                [],
            )
        )
        for model in processed_models
    ]

    exclusion_counts = [
        len(
            model.relationships.get(
                "exclusions",
                [],
            )
        )
        for model in processed_models
    ]

    dynamic_counts = [
        count_dynamic_value_dependencies(model)
        for model in processed_models
    ]

    table6_rows.append(
        {
            "Case": case_name,
            "Source": source_items,
            "Variants": len(processed_models),
            "Active": format_range(active_counts),
            "Reduction": format_percentage_range(reductions),
            "Implications": format_range(implication_counts),
            "Couplings": format_range(coupling_counts),
            "Exclusions": format_range(exclusion_counts),
            "Dynamic VBI": format_range(dynamic_counts),
            "Branch names": [
                branch_label(model)
                for model in processed_models
            ],
        }
    )


table6_observed = pd.DataFrame(table6_rows)

table6_observed

,Case,Source,Variants,Active,Reduction,Implications,Couplings,Exclusions,Dynamic VBI,Branch names
0,Motorola,35,1,35,0.00%,0,0,0,0,[Base]
1,Classic,20,1,18,10.00%,7,0,0,0,[Base]
2,Classic-100,100,1,96,4.00%,37,0,0,0,[Base]
3,WordProc-50,50,1,42,16.00%,63,0,0,7,[Base]
4,Concise_Case_11,12,2,8-10,16.67-33.33%,7-9,0,0,2-3,"[Excl(r7), Excl(r9)]"
5,Long_Case_30,30,3,20-22,26.67-33.33%,20-22,0,0,2-3,"[Excl(r9) ➔ Excl(r17), Excl(r9) ➔ Excl(r18), E..."


## Table 7: Operational execution with a compatible solver
Each retained source case is executed using the solver and configuration declared in Table 4.

For every returned front, the notebook records:

- raw outputs, when exposed by the solver;
- distinct retained release compositions;
- distinct objective vectors;
- feasibility;
- preservation of selected decision-item composition.

These cardinalities describe the retained configurations and must not be used for cross-case or cross-algorithm performance comparison.

In [10]:
table7_rows = []
table7_fronts = {}

for case_name, config in TABLE4_CONFIG.items():
    source_models = processed_case_models.get(case_name)

    if source_models is None:
        _, source_model = load_case(CASE_PATHS[case_name])

        source_models = PreprocessingEngine.process(
            initial_model=source_model,
            pipeline_options={},
        )

    selected_model = select_variant(
        source_models,
        config["variant"],
    )

    constraint_attribute = config["constraint_attribute"]

    constraints = {
        constraint_attribute: ConstraintSpec(
            attribute=constraint_attribute,
            operator=ThresholdOperator.LESS_EQUAL,
            value=config["constraint_value"],
        )
    }

    problem = OptimizationProblem(
        model=selected_model,
        objectives=config["objectives"],
        constraints=constraints,
    )

    solver = build_reported_solver(config)

    start_time = time.perf_counter()
    front = solver.solve(problem)
    execution_time = time.perf_counter() - start_time

    front.set_objective_directions(
        problem.get_objective_directions()
    )

    objective_names = list(config["objectives"].keys())

    plans = unique_release_compositions(front)
    vectors = unique_objective_vectors(
        front,
        objective_names,
    )

    all_feasible = all(
        solution.is_feasible
        for solution in front.solutions
    )

    composition_preserved = all(
        bool(solution.selected_ids)
        or len(solution.selected_ids) == 0
        for solution in front.solutions
    )

    table7_fronts[case_name] = front

    table7_rows.append(
        {
            "Case": case_name,
            "Solver": config["solver"],
            "Plans": len(plans),
            "Vectors": len(vectors),
            "Feasible": all_feasible,
            "Composition preserved": composition_preserved,
            "Execution time (s)": execution_time,
            "Variant": branch_label(selected_model),
        }
    )


table7_observed = pd.DataFrame(table7_rows)

table7_observed

,Case,Solver,Plans,Vectors,Feasible,Composition preserved,Execution time (s),Variant
0,Motorola,NSGA-II,38,32,True,True,33.430065,Base
1,Classic,Brute Force,25,25,True,True,6.858919,Base
2,Classic-100,NSGA-II,43,43,True,True,50.976602,Base
3,WordProc-50,NSGA-II,79,79,True,True,70.447046,Base
4,Concise_Case_11,Brute Force,21,21,True,True,0.018386,Excl(r7)
5,Long_Case_30,NSGA-II,20,20,True,True,66.713750,Excl(r9) ➔ Excl(r17)


## Objective vector

In [11]:
for case_name, front in table7_fronts.items():

    objective_names = list(
        TABLE4_CONFIG[case_name][
            "objectives"
        ].keys()
    )

    vectors = [
        tuple(
            round(
                float(
                    solution.objectives[
                        name
                    ]
                ),
                6,
            )
            for name
            in objective_names
        )
        for solution in front.solutions
    ]

    vector_counts = Counter(vectors)

    duplicates = {
        vector: count
        for vector, count
        in vector_counts.items()
        if count > 1
    }

    print(
        "\n",
        case_name,
    )

    print(
        "Plans:",
        len(
            unique_release_compositions(
                front
            )
        ),
    )

    print(
        "Objective vectors:",
        len(
            set(vectors)
        ),
    )

    print(
        "Repeated vectors:",
        duplicates
        if duplicates
        else "None",
    )


 Motorola
Plans: 38
Objective vectors: 32
Repeated vectors: {(13.0, 850.0): 2, (11.25, 610.0): 2, (8.5, 330.0): 2, (13.75, 940.0): 2, (7.5, 290.0): 2, (12.75, 800.0): 2}

 Classic
Plans: 25
Objective vectors: 25
Repeated vectors: None

 Classic-100
Plans: 43
Objective vectors: 43
Repeated vectors: None

 WordProc-50
Plans: 79
Objective vectors: 79
Repeated vectors: None

 Concise_Case_11
Plans: 21
Objective vectors: 21
Repeated vectors: None

 Long_Case_30
Plans: 20
Objective vectors: 20
Repeated vectors: None


## Table 8

In [12]:
_, controlled_model = load_case(
    CASE_PATHS[
        "Controlled_Linear_6"
    ]
)

controlled_models = (
    PreprocessingEngine.process(
        initial_model=controlled_model,
        pipeline_options={},
    )
)

controlled_model = controlled_models[0]

controlled_problem = (
    OptimizationProblem(
        model=controlled_model,
        objectives={
            "value": "max",
            "cost": "min",
        },
        constraints={
            "cost": ConstraintSpec(
                attribute="cost",
                operator=ThresholdOperator.LESS_EQUAL,
                value=24,
            )
        },
    )
)

## Exact front

In [13]:
reference_front = BruteForceSolver(
    max_evaluations=1000
).solve(
    controlled_problem
)

reference_front.set_objective_directions(
    controlled_problem.get_objective_directions()
)

reference_plans = (
    unique_release_compositions(
        reference_front
    )
)

reference_vectors = (
    unique_objective_vectors(
        reference_front,
        ["value", "cost"],
    )
)

print(
    "Reference plans:",
    len(reference_plans),
)

print(
    "Reference vectors:",
    len(reference_vectors),
)

Reference plans: 18
Reference vectors: 17


Because NSGA-II, Random Search, and Simulated Annealing are stochastic, the reported cardinalities represent a reference execution and may vary between runs. Exact agreement is only expected for Brute Force, where complete enumeration is performed.

In [20]:
PORTABILITY_SOLVERS = {
    "Brute Force":
        BruteForceSolver(
            max_evaluations=1000
        ),

    "Greedy":
        GreedySolver(
            weight_samples=5
        ),

    "NSGA-II":
        NSGA2Solver(
            pop_size=20,
            generations=20,
        ),

    "MILP":
        MILPSolver(
            weight_samples=4,
            time_limit_per_sweep=5.0,
            seed=42,
        ),

    "Random Search":
        RandomSearchSolver(
            num_samples=64
        ),

    "Simulated Annealing":
        SimulatedAnnealingSolver(
            iterations=300,
            initial_temp=20.0,
            cooling_rate=0.95,
            weight_samples=2,
        ),
}

table8_rows = []

for name, solver in (
    PORTABILITY_SOLVERS.items()
):

    front = solver.solve(
        controlled_problem
    )

    front.set_objective_directions(
        controlled_problem.get_objective_directions()
    )

    plans = (
        unique_release_compositions(
            front
        )
    )

    vectors = (
        unique_objective_vectors(
            front,
            ["value", "cost"],
        )
    )

    recovered_vectors = (
        vectors
        & reference_vectors
    )

    table8_rows.append(
        {
            "Algorithm": name,
            "Raw outputs": len(
                front.solutions
            ),
            "Plans": len(plans),
            "Objective vectors": len(
                vectors
            ),
            "Recovered reference vectors":
                f"{len(recovered_vectors)}/{len(reference_vectors)}",
        }
    )

table8_observed = pd.DataFrame(
    table8_rows
)

table8_observed

,Algorithm,Raw outputs,Plans,Objective vectors,Recovered reference vectors
0,Brute Force,18,18,17,17/17
1,Greedy,5,1,1,1/17
2,NSGA-II,11,11,11,11/17
3,MILP,3,1,1,1/17
4,Random Search,11,11,11,10/17
5,Simulated Annealing,1,1,1,1/17
